# 00 — Data contract, universe and correlation structure

The universe, dates and conventions every later notebook imports from `maplab.contract`; the no-look-ahead `Panel`; and the correlation facts behind the UUP-heavy weights of the risk-based methods. Nothing here allocates, and every statistic is descriptive.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import maplab as ml
import _nb00_helpers as h
from maplab import Panel, apply_style

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## §1 Universe and conventions

13 ETFs in 6 groups, one per risk premium; BIL (T-bills) is the risk-free rate and is never allocated. Every model reads these constants from `maplab.contract`, so no notebook can change them locally.

In [ ]:
groups = pd.Series(
    {g: ", ".join(t) for g, t in ml.ASSET_GROUPS.items()}
).rename("tickers").to_frame()
groups["n"] = [len(v) for v in ml.ASSET_GROUPS.values()]
print(f"Total universe: {len(ml.UNIVERSE)} instruments")
groups[["n", "tickers"]]

Conventions fixed once in the harness (`src/maplab/`); the constants are printed below.

- Returns: log returns for estimation, simple returns for compounding.
- Rebalancing: calendar month-end labels; every 252-day run starts at the first label with a full window, 2009-01-31, giving 208 rebalances, 40 of them after the 2022-12-31 split (notebook 09's longer lookbacks start later).
- Turnover: one-way, from the drifted weights to the new target, annualized by the calendar. Costs: 10 bps one-way, booked on the first trading day of each holding segment.
- Constraints: an attribute of each method, not a global rule.
- Missing data: forward-fill only, never back-fill.
- Continuity: every asset trades from the 2008-01-02 start; instruments not live by then are excluded (printed below).
- Benchmarks: EW and 60/40 (60% SPY, 40% IEF), rebalanced monthly to target like every other run; neither is registered.

In [ ]:
try:
    rf_full_ann = float(ml.load_rf_returns()["BIL"].mean() * ml.TRADING_DAYS)
    rf_note = f"{rf_full_ann:.2%} ann., full-window"
except FileNotFoundError:
    rf_note = "cache not built yet"

print("Window      :", ml.START, "→", ml.END)
print("Rebalance   :", ml.REBALANCE, "(month-end)")
print("Cov lookback:", ml.COV_LOOKBACK, "days")
print("Split       :", ml.TRAIN_TEST_SPLIT)
print("Risk-free   :", ml.RF_TICKER, f"({rf_note})")
print("Cost        :", f"{ml.COST_BPS:.0f} bps one-way")
print("Excluded    :", ml.contract.EXCLUDED_FOR_CONTINUITY, "(continuity)")

## §2 Data and the Panel

Prices come from one cached file, `data/cache/prices.parquet`, which `scripts/build_panel.py` builds on SPY's trading calendar with forward-fill only. `ml.load_prices()` returns the 13 allocatable assets; BIL is read separately by `ml.load_rf_returns()`. Without the cache, the cell below builds a synthetic panel on the same calendar, for plumbing checks only.

In [ ]:
USING_SYNTHETIC = False
try:
    prices = ml.load_prices()
    print("Loaded REAL cached panel.")
except FileNotFoundError:
    USING_SYNTHETIC = True
    rng = np.random.default_rng(0)
    idx = pd.bdate_range(ml.START, ml.END)
    n = len(ml.UNIVERSE)
    mu = rng.normal(0.06, 0.04, n) / ml.TRADING_DAYS
    vol = rng.uniform(0.10, 0.40, n) / np.sqrt(ml.TRADING_DAYS)
    # light common factor so correlations aren't trivially zero
    common = rng.standard_normal((len(idx), 1)) * 0.4
    shocks = rng.standard_normal((len(idx), n)) * 0.6 + common
    rets = pd.DataFrame(mu + vol * shocks, index=idx, columns=ml.UNIVERSE)
    prices = 100 * np.exp(rets.cumsum())
    print("⚠️  SYNTHETIC fallback in use — for plumbing review only, not results.")

print(f"prices: {prices.shape[0]:,} days × {prices.shape[1]} assets "
      f"({prices.index.min().date()} → {prices.index.max().date()})")

Log returns feed estimation, simple returns the backtest. `panel.slice(asof, …)` returns only rows dated strictly before `asof`; the cell asserts it at one label.

In [ ]:
log_returns = ml.to_log_returns(prices)    # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
panel = Panel({"prices": prices, "returns": log_returns})

# demonstrate the gatekeeper
asof = pd.Timestamp("2020-03-31")
window = panel.slice(asof, "returns", lookback=ml.COV_LOOKBACK)
print(f"slice(asof={asof.date()}, lookback=252):")
print(f"  rows: {len(window)}  |  last available date: {window.index[-1].date()}")
assert window.index.max() < asof, "no-look-ahead violated!"
print("  ✓ all data strictly before asof")

The cell below also writes log and simple returns to `data/cache/` as a convenience copy; the model notebooks build their returns from the prices.

In [ ]:
cache = Path.cwd().parent / "data" / "cache"
cache.mkdir(parents=True, exist_ok=True)
if USING_SYNTHETIC:
    prices.to_parquet(cache / "prices.parquet")
else:
    # Real prices.parquet already carries 13 assets + BIL via
    # scripts/build_panel.py — don't overwrite it with `prices` (13 only,
    # from ml.load_prices()), which would silently drop BIL from the cache.
    print("Skipping prices.parquet re-cache: real cache (13 assets + BIL) "
          "is already authoritative via scripts/build_panel.py.")
log_returns.to_parquet(cache / "returns_log.parquet")
simple_returns.to_parquet(cache / "returns_simple.parquet")
print("cached:", sorted(p.name for p in cache.glob("*.parquet")))

## §3 Correlation statistics

At each rebalance label $t$ the statistics use the models' window: the 252 daily log returns dated strictly before $t$, with Pearson correlations $\rho_{ij}(t)$. With $U$ = UUP and $R$ the other 12 assets,

$$\bar\rho_U(t) = \frac{1}{12}\sum_{j \in R} \rho_{Uj}(t), \qquad \sigma_U(t) = \sqrt{252}\;\mathrm{sd}_t(r_U), \qquad \bar\rho_{\mathrm{rest}}(t) = \frac{1}{12 \cdot 11}\sum_{i \in R}\,\sum_{j \in R,\, j \neq i} \rho_{ij}(t).$$

GMV and MDP weights depend on UUP's correlations (notebooks 01 and 05); $\bar\rho_U$ is their average. HRP gives UUP, when it splits off alone at the root, the weight $V_{\mathrm{rest}}/(\sigma_U^2 + V_{\mathrm{rest}})$, where $V_{\mathrm{rest}}$ is the variance of the inverse-variance portfolio of the other 12 (notebook 07). With their volatilities fixed, $V_{\mathrm{rest}}$ rises with every correlation among the 12, and $\bar\rho_{\mathrm{rest}}$ is their average. The full-sample versions in §4 use every day from 2008-01-03 to 2026-04-30.

## §4 Full-sample structure

Coverage and mean annualized volatility by group, then the full-sample correlation matrix.

In [ ]:
coverage = prices.notna().mean().rename("coverage")
ann_vol = (log_returns.std() * np.sqrt(ml.TRADING_DAYS)).rename("ann_vol")
diag = pd.concat([coverage, ann_vol], axis=1)
diag["group"] = [ml.GROUP_OF[t] for t in diag.index]
diag.groupby("group").agg(
    n=("coverage", "size"),
    min_coverage=("coverage", "min"),
    mean_ann_vol=("ann_vol", "mean"),
).round(3)

Coverage is 1.000 in every group: no asset starts after 2008-01-02. Mean annualized volatility runs from 6.3% (inflation-linked) to 23.2% (equity); UUP, the whole fx group, is at 8.1%, the second-lowest of the 6.

The correlation matrix in the leaf order of notebook 07's clustering (single linkage on the distance of distances), values annotated, UUP's row and column boxed.

In [ ]:
fig, order_tickers = h.clustered_corr_figure(log_returns)
plt.show()
print(f"leaf order (full sample, nb07 recipe): {order_tickers}")

uup_corr = log_returns[ml.UNIVERSE].corr()["UUP"].drop("UUP").sort_values().rename_axis(None)
print("UUP full-sample correlations with the other 12:")
print(uup_corr.to_string(float_format=lambda v: f"{v:.4f}"))

All 12 of UUP's full-sample correlations are negative, from −0.07 with TLT to −0.42 with GLD and EFA; no other asset is negatively correlated with every other asset. UUP is first in the leaf order. Full-sample and descriptive: no allocation in this repo uses a full-sample estimate.

## §5 UUP and the rest through time

The §3 statistics at the 208 rebalance labels, then their medians over the train (168 labels, to 2022-12-31) and test (40 labels) windows and their 2022-12-31 values.

In [ ]:
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"

fig, uup_table = h.uup_rolling_figure(panel, rdates, split_ts)
plt.show()

uup_summary = pd.DataFrame({
    "train (<= split)": uup_table.loc[uup_table.index <= split_ts].median(),
    "test (> split)": uup_table.loc[uup_table.index > split_ts].median(),
    "2022-12-31": uup_table.loc[pd.Timestamp("2022-12-31")],
}).T
print(uup_summary.to_string(float_format=lambda v: f"{v:.4f}"))

After 2022 two things move: UUP's median correlation with the other 12 falls from −0.2553 to −0.4511, and ρ̄_rest rises from 0.2409 to 0.4440. σ_UUP barely changes (0.0752 → 0.0741). The first reaches the methods that read UUP's correlations (GMV, MDP); the second is the route by which HRP's corner returns (notebook 07). At 2022-12-31 the three are −0.4659 (−0.47 in notebook 05), 0.0927 (9.27% in notebook 04) and 0.4511.

## §6 TAKEAWAY — UUP is negatively correlated with every other asset, and the other 12 became more correlated after 2022

- **One contract.** 13 ETFs in 6 groups, BIL as the risk-free rate, month-end rebalancing on a trailing 252-day window, 10 bps one-way, split at 2022-12-31: 208 rebalances, 40 in the test window. EW and 60/40 are the benchmarks; neither is registered.
- **No look-ahead by construction.** `Panel.slice` returns only rows dated strictly before the label; estimation uses log returns, compounding simple returns.
- **UUP is the exception in the full sample.** Its 12 correlations run from −0.07 to −0.42; no other asset is negatively correlated with every other.
- **After 2022 two things move.** UUP's median correlation with the rest falls from −0.2553 to −0.4511, ρ̄_rest rises from 0.2409 to 0.4440, and σ_UUP barely changes (0.0752 → 0.0741).
- **What they feed.** GMV and MDP read UUP's correlations; HRP's root split compares σ_UUP with the variance of the other 12 (notebook 07); ρ̄_rest is the candidate regime variable for Phase 3.

The UUP-heavy weights of the risk-based methods start from one asset negatively correlated with all the others, in a book whose other 12 assets moved together more after 2022.